# 🧠 Attention Models in Computer Vision

**Attention mechanisms** have revolutionized deep learning by allowing models to focus on the most relevant parts of the input data. In computer vision, this means a model can intelligently "look at" specific regions of an image when making a prediction or generating a description. This notebook will break down the key concepts and applications of attention.

-----

## 🧐 Introduction to Attention Models in Vision

Standard Convolutional Neural Networks (CNNs) process an entire image at once, which can be inefficient and lead to a loss of important information. An attention mechanism addresses this by creating a **context vector** that is a weighted sum of input features. These weights are learned by the model, effectively telling it which parts of the image are most important for the task at hand.

The core idea is to let the model decide **what to focus on**. For instance, when an image captioning model is generating the word "dog," its attention mechanism can learn to focus on the pixels of the dog in the image.

-----

## 🗣️ Vision and Language

Attention models are particularly powerful in tasks that combine visual and textual data.

### **1. Image Captioning**

**Image captioning** is the task of generating a natural language description for an image. It’s typically solved with an **Encoder-Decoder** architecture.

  * **Encoder:** A CNN (e.g., ResNet) that processes the image and extracts a set of feature vectors, one for each spatial region.
  * **Decoder:** An RNN (e.g., LSTM) that generates the caption word by word.

**How Attention Helps:** At each step of the caption generation, the decoder uses an attention mechanism to look back at the encoded image features. It calculates attention weights, which are essentially a probability distribution over the image regions. The weighted sum of these features creates a context vector that guides the next word prediction.

#### **Conceptual Code for Attention-based Image Captioning**

```python
import tensorflow as tf
from tensorflow.keras.layers import Layer, AdditiveAttention, Dense, LSTM

# Conceptual model for Attention-based Image Captioning
class BahdanauAttention(Layer):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = Dense(units)
        self.W2 = Dense(units)
        self.V = Dense(1)

    def call(self, features, hidden):
        # features shape: (batch_size, 64, 2048)
        # hidden shape: (batch_size, 512)
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        # score shape: (batch_size, 64, 1)
        score = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))

        # attention_weights shape: (batch_size, 64, 1)
        attention_weights = tf.nn.softmax(self.V(score), axis=1)

        # context_vector shape: (batch_size, 2048)
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

# The full model would combine this attention layer with an encoder (e.g., CNN)
# and a decoder (e.g., LSTM). The LSTM would get the context vector
# and the previous word embedding as input to predict the next word.
```

-----

### **2. Visual QA (VQA)**

**Visual Question Answering** is the task of answering a natural language question about an image. Attention is used here to fuse information from the question and the image. The model learns to jointly attend to the most relevant parts of the image **and** the most relevant words in the question to arrive at the correct answer.

### **3. Visual Dialog**

**Visual Dialog** is an extension of VQA where a model has a conversation with a human about an image. Attention is crucial for tracking the dialog history and deciding which parts of the image and previous turns in the conversation are most relevant to the current question.

-----

## 🗺️ Spatial Transformers

A **Spatial Transformer Network (STN)** is a differentiable module that can be added to any CNN. It allows the network to actively learn **geometric transformations** (e.g., rotation, scaling, translation) on the input image or feature map. This makes the model invariant to these transformations, improving performance.

**How it works:**

1.  **Localization Network:** A small network (often a few layers) that takes a feature map and regresses the parameters of a transformation (e.g., rotation angle, translation).
2.  **Grid Generator:** Takes the transformation parameters and creates a sampling grid.
3.  **Sampler:** Warps the input feature map according to the sampling grid, producing an output feature map that is geometrically aligned.

#### **Code for a Spatial Transformer Layer**

```python
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from keras.layers import Input
from keras.models import Model
from tensorflow.keras.backend import spatial_transformer_2d

# Define the Spatial Transformer Layer
class SpatialTransformer(Layer):
    def __init__(self, localization_network):
        super(SpatialTransformer, self).__init__()
        self.localization_network = localization_network

    def build(self, input_shape):
        self.loc_model = self.localization_network
        # The output of the localization network is a 6-parameter affine transformation
        self.loc_model.add(Dense(6, activation='linear',
                                 kernel_initializer='zeros',
                                 bias_initializer=tf.keras.initializers.constant([1.0, 0.0, 0.0, 0.0, 1.0, 0.0])))

    def call(self, inputs):
        # Get the transformation parameters
        theta = self.loc_model(inputs)
        # Apply the transformation to the input feature map
        return spatial_transformer_2d(inputs, theta)

# Example of a simple localization network
loc_net = Sequential([
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(20, activation='relu')
])

# Create a model with the Spatial Transformer Layer
input_shape = (64, 64, 1)
inputs = Input(input_shape)
stn_layer = SpatialTransformer(loc_net)(inputs)
x = Conv2D(64, (3, 3), activation='relu')(stn_layer)
# ... rest of your CNN
```

**Note:** `spatial_transformer_2d` is a function from `keras-contrib` and requires installation. This code block is a simplified conceptual example.

-----

## 🤖 Transformer Networks

**Transformer Networks** were originally developed for natural language processing and are based entirely on **self-attention**, without using any recurrence or convolution. They have since been adapted for computer vision with the **Vision Transformer (ViT)**.

### **Key Concepts:**

  * **Self-Attention:** A mechanism that allows a network to weigh the importance of different elements in a sequence to each other. For an image, this means each image patch can attend to every other patch.
  * **Multi-Head Attention:** A technique that performs self-attention multiple times in parallel ("multiple heads"). Each head can learn to focus on different aspects of the input, leading to a richer representation.
  * **Encoder-Decoder Structure:** The original Transformer has a stacked encoder and decoder. The ViT, however, uses only the encoder.

#### **Vision Transformer (ViT) Overview**

The ViT model works by:

1.  **Splitting the image into patches:** The image is divided into a grid of non-overlapping patches.
2.  **Linear Embedding:** Each patch is flattened and linearly embedded into a feature vector.
3.  **Position Embeddings:** To preserve spatial information, positional embeddings are added to the patch embeddings.
4.  **Transformer Encoder:** The sequence of patch embeddings is fed into a standard Transformer encoder, which uses a series of multi-head self-attention and feed-forward layers.
5.  **Classification Head:** A final classification head (e.g., a simple MLP) is used to predict the class label.

#### **Conceptual Code for a Vision Transformer (ViT)**

```python
import tensorflow as tf
from tensorflow.keras.layers import Layer, Dense, Dropout, LayerNormalization

# A simplified self-attention block
class MultiHeadSelfAttention(Layer):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.num_heads = num_heads
        self.embed_dim = embed_dim
        assert embed_dim % num_heads == 0
        self.depth = embed_dim // num_heads

        self.wq = Dense(embed_dim)
        self.wk = Dense(embed_dim)
        self.wv = Dense(embed_dim)
        self.dense = Dense(embed_dim)

    def call(self, q, k, v):
        # Conceptual call method, handles splitting and concat
        # ... logic for multi-head self-attention ...
        return self.dense(v) # Simplified for brevity

# A single Transformer block
class TransformerBlock(Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = MultiHeadSelfAttention(embed_dim, num_heads)
        self.ffn = Sequential([Dense(ff_dim, activation="relu"), Dense(embed_dim)])
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(rate)
        self.dropout2 = Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# The full ViT would be a series of these blocks applied to image patches.
# This code is a conceptual example of the core components.
```

https://github.com/codebasics/deep-learning-keras-tf-tutorial/blob/master/22_word_embedding/supervised_word_embeddings.ipynb

In [1]:
https://www.linkedin.com/pulse/iris-dataset-analysis-using-machine-learning-techniques-pramod-sahu-g3kgf/

SyntaxError: invalid syntax (ipython-input-3407281797.py, line 1)

In [2]:
https://www.tensorflow.org/text/tutorials/transformer

SyntaxError: invalid syntax (ipython-input-2989204411.py, line 1)